# Exercises XP: Day 2
Follow the instructions. Where you see TODO, add your answer before running/continuing.


## What You'll Learn
- Deepen your understanding of core LLM concepts.
- Apply theory to practical scenarios.
- Develop critical thinking on LLM applications and ethics.
- Compare/contrast transformer architectures and techniques.


## What You Will Create
- Comparative tables (NLP paradigms and BERT variants)
- Architecture/application write-ups
- Pretraining benefits and ethical considerations
- Analyses on attention and positional encoding
- Model selection justifications across tasks
- Notes on softmax temperature and its effects
- Scenario-based answers applying learned concepts


## 🌟 Exercise 1: Traditional vs. Modern NLP: Comparative Analysis
1) Complete the table (replace each TODO):

| Aspect | Traditional NLP | Modern NLP |
|---|---|---|
| Feature Engineering | Manual, hand-crafted features (bag-of-words, n-grams, TF-IDF, POS tags, hand-written rules) designed by domain experts | Automatic, learned representations — the model discovers features end-to-end during training, little to no manual engineering |
| Word Representations | Sparse, high-dimensional, discrete (one-hot vectors, count/TF-IDF matrices) with no notion of meaning or similarity | Dense, low-dimensional, continuous embeddings (Word2Vec, GloVe, and contextual embeddings from BERT/GPT) that capture semantic similarity |
| Model Architectures | Linear/statistical models: Naïve Bayes, SVM, logistic regression, HMMs, CRFs, decision trees | Deep neural networks: RNNs/LSTMs/GRUs and especially Transformers (self-attention) |
| Training Methodology | Task-specific supervised models trained from scratch on a single labeled dataset for one task | Self-supervised pre-training on huge unlabeled corpora, then fine-tuning / prompting for downstream tasks (transfer learning) |
| Key Examples | Naïve Bayes spam filter, TF-IDF + SVM text classifier, CRF-based NER, n-gram language models | BERT, GPT family, T5, RoBERTa, LLaMA |
| Advantages | Fast, cheap, interpretable, works on small data, low compute, easy to debug | State-of-the-art accuracy, captures context and long-range/semantic relations, transferable across tasks, minimal feature engineering |
| Disadvantages | Brittle, needs expert feature design, ignores word order/context, poor generalization, weak on semantics | Very compute/memory hungry, data-hungry to pre-train, hard to interpret ("black box"), risk of inheriting bias from training data |

2) Discuss: How did the shift to modern NLP impact scalability and efficiency?

The move to learned, dense representations meant engineers no longer had to hand-craft features for every new task, so the same pre-trained model could be reused across many problems — this dramatically improved *developer* efficiency and scalability. The Transformer architecture replaced sequential recurrence with self-attention, which is fully parallelizable and maps naturally onto modern GPU/TPU hardware, allowing training on far larger datasets in less wall-clock time. Transfer learning changed the economics: one expensive self-supervised pre-training run is amortized over countless cheap fine-tuning runs, so most teams only pay the small downstream cost. This also slashed the amount of labeled data needed per task, since the model already encodes general language knowledge. The trade-off is that the pre-training stage itself is enormously compute- and energy-intensive, and inference for billion-parameter models is heavy, driving demand for compression techniques (distillation, quantization, pruning) and parameter-efficient fine-tuning. On balance, modern NLP scaled *capability and reuse* up sharply while pushing the heavy cost into a one-time pre-training phase that the wider community shares.


## 🌟 Exercise 2: LLM Architecture and Application Scenarios
For each, describe (a) core architectural differences, (b) a real application, (c) why it fits.

### BERT
- Architecture: Bidirectional **encoder-only** Transformer; pre-trained with Masked Language Modeling (MLM) — and historically Next Sentence Prediction (NSP). Because it attends to the whole sentence in both directions at once, every token's representation is informed by both its left and right context, which makes it excellent at *understanding* text but not at generating it.
- Application: Sentiment analysis / text classification, Named-Entity Recognition (NER), and extractive question answering (e.g., powering Google Search query understanding).
- Why: These tasks need deep *understanding* of a complete, already-given input rather than producing new text. Bidirectional context lets BERT disambiguate words from both sides (e.g., "bank" of a river vs. a "bank" account), which is exactly what classification/NER/extractive-QA require.

### GPT
- Architecture: **Decoder-only**, autoregressive Transformer with a causal (left-to-right) attention mask; pre-trained with Causal Language Modeling (CLM) — predict the next token given all previous ones. Each token may only attend to earlier tokens, which is what enables coherent generation.
- Application: Conversational assistants/chatbots, open-ended text and code generation (ChatGPT, GitHub Copilot).
- Why: The next-token objective and causal masking make GPT naturally suited to *producing* fluent, contextually consistent text one token at a time. Its in-context / few-shot ability also lets it adapt to new tasks from a prompt without retraining.

### T5
- Architecture: **Encoder–decoder** (full seq2seq) Transformer that casts *every* NLP problem into a unified **"text-to-text"** format — input text in, output text out — pre-trained with a span-corruption (denoising) objective.
- Application: Summarization, machine translation, and other transformation tasks (e.g., "translate English to German: ...", "summarize: ..."). 
- Why: Tasks that map one piece of text to a *different* piece of text need both an encoder to fully understand the source and a decoder to generate the target. The text-to-text framing means a single model and training recipe handle many tasks just by changing the task prefix, which is ideal for translation/summarization.


## 🌟 Exercise 3: Benefits and Ethics of Pre-training
- Benefits (explain each in your words):
  1) Improved generalization: By learning from a huge, diverse corpus, the model absorbs broad patterns of grammar, facts, and meaning. This general knowledge transfers to inputs and tasks it never saw explicitly, so it performs better on unseen data than a model trained from scratch on one small dataset.
  2) Less labeled data: Pre-training is *self-supervised* — it uses raw text with no human labels (the text itself provides the targets). The downstream task then only needs a small labeled set to fine-tune, because the model already understands language; this is huge where labeling is expensive (medical, legal).
  3) Faster fine-tuning: Starting from pre-trained weights means the model is already near a good region of parameter space, so adapting to a new task converges in a few epochs instead of training a deep network from random initialization.
  4) Transfer learning: One pre-trained backbone can be reused for many different downstream tasks (classification, NER, QA, summarization) just by swapping/fine-tuning a small head, amortizing the cost of pre-training across applications.
  5) Robustness: Exposure to varied text (different domains, styles, noise, spellings) makes the model more resilient to input variation and less likely to break on slightly out-of-distribution data than a narrowly trained model.

- Ethical concerns: 
  - **Bias** — models absorb and can amplify social, gender, racial, and cultural biases present in the training data.
  - **Misinformation / hallucination** — they can generate fluent but false or fabricated statements that sound authoritative.
  - **Misuse** — can be weaponized for spam, phishing, propaganda, deepfake text, plagiarism, or automated abuse at scale.
  - **Privacy** — training data scraped from the web may contain personal information that the model can memorize and leak.
  - **Environmental / access cost** — large-scale pre-training consumes significant energy and concentrates capability in a few well-resourced organizations.

- Mitigations: 
  - **Data curation & filtering** — clean, deduplicate, and balance training data; remove PII and toxic content; document datasets (datasheets / model cards).
  - **Differential privacy** and deduplication to limit memorization of individual records.
  - **Safety filters / guardrails** — content moderation classifiers on inputs and outputs.
  - **RLHF / alignment** — fine-tune with human feedback to make outputs more helpful, honest, and harmless.
  - **Audits & evaluation** — bias and fairness benchmarks, red-teaming, transparency reports, and human oversight before deployment.


## 🌟 Exercise 4: Transformer Architecture Deep Dive
### Self-Attention & Multi-Head Attention
- **Q/K/V flow, softmax weighting, value mixing:** Each input token's embedding is linearly projected into three vectors — a **Query (Q)** ("what am I looking for?"), a **Key (K)** ("what do I offer?"), and a **Value (V)** ("the content I carry"). The model takes the dot product of each query with every key to get a relevance score, divides by √dₖ (scaling) to keep gradients stable, and applies **softmax** across the keys so the scores become attention weights that sum to 1. The output for each token is the **weighted sum of all Value vectors** using those weights — so each token's new representation is a blend of the information from the tokens it found most relevant.
- **Why multiple heads help:** Multi-head attention runs several attention operations in parallel, each with its *own* learned Q/K/V projections into a different subspace. Each head can therefore focus on a different kind of relationship (syntax, coreference, long-range dependency, local phrasing) simultaneously, and their outputs are concatenated and projected back. This gives the model a richer, multi-perspective view than a single head, and averages out noise from any one projection.
- **Example sentence (different from lesson):**
  - "The tired developer fixed the bug because **it** kept crashing the server overnight."
  - One head might capture **coreference**: linking the pronoun "it" back to "the bug" (what was crashing). A different head might capture a **syntactic subject–verb relation**: tying "developer" to its verb "fixed". A third could capture a **causal/long-range** link between "fixed" and "because ... crashing." Distinct heads attend to these different relations at the same time.

### Pre-training Objectives
- **MLM vs CLM:** *Masked Language Modeling* (MLM, used by BERT) randomly masks some tokens and trains the model to predict them using **bidirectional** context (both left and right) — great for *understanding*. *Causal Language Modeling* (CLM, used by GPT) predicts the **next** token using only **left** context — great for *generation*.
- **When to prefer MLM vs CLM:** Prefer **MLM** when the goal is understanding/encoding a complete input — classification, NER, extractive QA, sentence embeddings. Prefer **CLM** when the goal is **generating** text — chatbots, autocompletion, story/code generation.
- **NSP:** *Next Sentence Prediction* trained original BERT to judge whether sentence B actually follows sentence A, aiming to teach inter-sentence/discourse relationships (helpful for QA and NLI). Many modern models (e.g., RoBERTa) **drop NSP** because later studies found it weak/noisy and showed that removing it — and training longer on full-length sequences — actually improves downstream performance.

### Transformer Model Selection
- **Sentiment on reviews:** **Encoder-only** (e.g., BERT/RoBERTa/DistilBERT). Justification: it's a *understanding/classification* task over a complete given text; bidirectional context yields the best representation and you just add a classification head — no generation needed.
- **Conversational chatbot (creative responses):** **Decoder-only** (e.g., GPT-style). Justification: the task is open-ended autoregressive *generation*; causal LMs produce fluent, coherent, creative text token-by-token and adapt via prompting.
- **Technical document translation (EN→ES):** **Encoder–Decoder** (e.g., T5 / MarianMT). Justification: translation maps one full sequence to a *different* sequence — the encoder fully understands the English source while the decoder generates the Spanish target, which seq2seq architectures are designed for.

### Positional Encoding
- **Purpose:** Self-attention is **permutation-invariant** — on its own it treats the input as an unordered *set* of tokens and has no notion of order or distance. Positional encodings (absolute sinusoidal/learned, or relative/RoPE) inject information about each token's position so the model can use word order and proximity, which carry meaning in language.
- **Example failure without positions:** Without position information, "The dog bit the man" and "The man bit the dog" contain the exact same tokens, so the model would produce identical representations and could not tell who did the biting — completely losing the meaning that depends on word order.


## 🌟 Exercise 5: BERT Variations: Choose Your Detective
Assign the best fit and justify:
- Scenario 1 (mobile, limited resources): **DistilBERT** — distilled to ~40% smaller and ~60% faster than BERT while keeping ~97% of its performance, ideal for on-device/low-latency use.
- Scenario 2 (legal docs, high accuracy): **RoBERTa** — robustly optimized BERT (more data, longer training, no NSP, dynamic masking) delivers top accuracy where correctness matters more than size.
- Scenario 3 (multilingual support): **XLM-R (XLM-RoBERTa)** — trained on 100 languages with cross-lingual transfer, the natural choice for multilingual tasks.
- Scenario 4 (efficient pretraining with token replacement detection): **ELECTRA** — its replaced-token-detection objective trains over *all* tokens, giving strong results with far less compute.
- Scenario 5 (efficient NLP, constrained environments): **ALBERT** — parameter sharing + factorized embeddings shrink memory dramatically, fitting tight-resource settings (DistilBERT is also valid if raw speed is the priority).

Create/completed table:

| Model | Training differences | Size/Efficiency | Innovations | Ideal use cases |
|---|---|---|---|---|
| RoBERTa | Removes NSP; trains on much more data for longer with larger batches; uses **dynamic** masking | Same size as BERT (no smaller), but better accuracy per parameter | Showed BERT was undertrained; dynamic masking; full-sentence training without NSP | High-accuracy understanding tasks where compute/size is not the constraint (classification, NLI, QA) |
| ALBERT | Same MLM/SOP objective but replaces NSP with **Sentence-Order Prediction** | Much smaller in parameters via **cross-layer parameter sharing** + **factorized embedding** parameterization | Parameter sharing and embedding factorization to cut memory; SOP objective | Memory-constrained settings needing BERT-level quality; large models that must fit limited RAM |
| DistilBERT | **Knowledge distillation** from BERT (student mimics teacher) during pre-training | ~40% smaller, ~60% faster, retains ~97% of BERT's performance | Triple distillation loss (MLM + distillation + cosine embedding) | Mobile/edge/real-time inference where speed and footprint matter |
| ELECTRA | **Replaced Token Detection** — a generator corrupts tokens, the discriminator predicts which were replaced (learns from all tokens, not just ~15%) | Far more **compute-efficient** to pre-train; strong results at small sizes | Sample-efficient discriminative pre-training objective | Efficient pre-training from scratch on a limited compute budget; strong small models |
| XLM-R | RoBERTa-style training on a massive **multilingual** CommonCrawl corpus (100 languages) | Large, but one model covers many languages (avoids per-language models) | Cross-lingual representation learning + zero-shot cross-lingual transfer | Multilingual and cross-lingual tasks, low-resource languages |


## 🌟 Exercise 6: Softmax Temperature: The Randomness Regulator
### 1) Temperature Scenarios
Temperature `T` rescales the logits (`logits / T`) before softmax: low T sharpens the distribution (more deterministic), high T flattens it (more random).
- T=0.2: **Very focused / near-deterministic.** The probability mass concentrates on the single most likely token, so output is conservative, repetitive, and "safe" — almost always picks the top choice. Good for factual, precise answers.
- T=1.5: **Highly random / creative.** The distribution is flattened, so unlikely tokens get a real chance. Output is diverse, surprising, and more original — but also more prone to incoherence, going off-topic, or factual errors.
- T=1.0: **Neutral / unscaled.** Samples directly from the model's true predicted probabilities — a balanced mix of coherence and variety (the default).

### 2) Application Design
- Bedtime stories (creativity vs coherence): Use a **moderately high temperature (~1.0–1.2)**, optionally combined with top-p (nucleus) sampling (~0.9). Why: stories benefit from imaginative, varied wording so they don't feel repetitive, but you still need enough coherence for the plot to make sense — a slightly-above-neutral temperature gives creative-but-readable text, while top-p clips the truly absurd tokens.
- Financial report summaries (accuracy/reliability): Use a **very low temperature (~0.0–0.3)**, effectively greedy/near-greedy decoding. Why: financial summaries must be accurate, factual, and reproducible — you want the model's most confident, consistent output and to minimize hallucination or random embellishment. Determinism is a feature here, not a limitation.

### 3) Temperature & Bias
Temperature changes *which* tokens get sampled, so it can surface or dampen biases present in the model. At **low temperature**, the model almost always emits its single most probable continuation; if the training data biased that top choice (e.g., defaulting "the nurse... **she**", "the engineer... **he**"), low T will *reliably reproduce* that stereotype every time, making the bias more visible and consistent. At **high temperature**, the distribution flattens, so less-likely (counter-stereotypical) tokens like "the nurse... **he**" get sampled more often — this can *dampen* the appearance of a single dominant bias by adding variety, but it can also surface rarer toxic or biased completions that normally sit in the low-probability tail. **Realistic example:** prompting "The CEO walked into the room and ___" at T=0.2 may almost always continue with "he," exposing a gender bias deterministically; at T=1.5 the pronoun varies more (sometimes "she"/"they"), reducing the *consistency* of the bias but occasionally producing unexpected or offensive completions. Key point: temperature doesn't remove bias — the bias lives in the learned probabilities — it only changes how often and how visibly that bias shows up.


### (Optional) Quick Generation Demo Across Temperatures
Note: Requires `transformers` and model download; skip if offline.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "gpt2"                       # chosen causal (decoder-only) LM for generation
prompt = "Artificial intelligence will"   # our seed text; the model continues from here

# Build a text-generation pipeline that ties the model + tokenizer together.
pipe = pipeline(
    "text-generation",
    model=AutoModelForCausalLM.from_pretrained(model_name),
    tokenizer=AutoTokenizer.from_pretrained(model_name),
    device=0 if torch.cuda.is_available() else -1,  # GPU (id 0) if available, else CPU (-1)
)

# Sample 40 new tokens at three temperatures to see how randomness/creativity scales.
# do_sample=True is required for temperature to have any effect (otherwise it's greedy).
for T in [0.2, 1.0, 1.5]:
    out = pipe(prompt, max_new_tokens=40, temperature=T, do_sample=True)
    print("--- Temperature:", T)
    print(out[0]["generated_text"])  # low T -> focused/repetitive, high T -> diverse/creative
